In [ ]:
# 1. 安装 Kaggle CLI
!pip install kaggle -q

# 2. 上传你的 kaggle.json（从 Kaggle → Account → Create API Token 下载）
from google.colab import files
files.upload()  # 选择你的 kaggle.json

# 3. 配置权限
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 4. 下载数据集
!kaggle datasets download -d shubhamkarande13/d-fire

# 5. 解压到指定文件夹
!unzip d-fire.zip -d /content/D-Fire

!pip install ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/DiffPure_extracted/DiffPure-master')  # 换成你实际放purify_bld_fixed.py的目录
import purify_bld_fixed as bld

from ultralytics import YOLO
detector = YOLO('/content/drive/MyDrive/dfire_checkpoints/yolov8n_dfire_detector.pt')

In [ ]:
CKPT_AE   = "/content/drive/MyDrive/bld_purify_checkpoints/bld_autoencoder.pt"
CKPT_DIFF = "/content/drive/MyDrive/bld_purify_checkpoints/bld_diffusion.pt"
import torch
ae = bld.BinaryAutoEncoder().to(bld.DEVICE)
ae.load_state_dict(torch.load(CKPT_AE, map_location=bld.DEVICE))
ae.eval()

diff_model = bld.ReverseModel(D=ae.D).to(bld.DEVICE)
diff_model.load_state_dict(torch.load(CKPT_DIFF, map_location=bld.DEVICE))
diff_model.eval()

betas, flip_prob = bld.get_schedule()
print("AE + diffusion 权重加载完毕")

In [ ]:
import torch
from ultralytics.utils.loss import v8DetectionLoss
detector.model = detector.model.to(bld.DEVICE)
detector.model.train()   # 切train模式，loss才会被正确计算（但不会更新权重，因为我们不会调用optimizer.step）
attack_yolo = YOLO('/content/drive/MyDrive/dfire_checkpoints/yolov8n_dfire_detector.pt')
attack_yolo.model.to(bld.DEVICE)          # ← 补上这一行
attack_yolo.model.train()
for p in attack_yolo.model.parameters():
    p.requires_grad_(True)

criterion = v8DetectionLoss(attack_yolo.model)


def pgd_attack(model, criterion, x, targets, eps=8/255, alpha=2/255, steps=10):
    """
    model   : detector.model (DetectionModel)
    x       : (B,3,H,W) float [0,1]，注意要 resize 到训练时的 imgsz=384
    targets : dict，需包含 'cls', 'bboxes', 'batch_idx' 三个key（YOLO格式）
    """
    x_adv = (x + torch.empty_like(x).uniform_(-eps, eps)).clamp(0, 1).requires_grad_(True)
    for _ in range(steps):
        preds = model(x_adv)
        loss, _ = criterion(preds, targets)
        loss = loss.sum()
        grad = torch.autograd.grad(loss, x_adv)[0]
        x_adv = (x_adv.detach() + alpha * grad.sign()).clamp(x - eps, x + eps).clamp(0, 1)
        x_adv.requires_grad_(True)
    return x_adv.detach()

print(criterion)
print(type(criterion))

In [ ]:
import glob
from PIL import Image
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

class DFireDetDataset(Dataset):
    def __init__(self, root, split='test', size=384):
        self.img_paths = sorted(glob.glob(f'{root}/{split}/images/*.jpg'))
        self.label_dir = f'{root}/{split}/labels'
        self.tf = T.Compose([T.Resize((size, size)), T.ToTensor()])

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        p = self.img_paths[idx]
        x = self.tf(Image.open(p).convert('RGB'))
        lp = os.path.join(self.label_dir, os.path.basename(p).rsplit('.', 1)[0] + '.txt')
        boxes = []
        if os.path.exists(lp):
            with open(lp) as f:
                for line in f:
                    parts = line.split()
                    if len(parts) == 5:
                        boxes.append([float(v) for v in parts])
        boxes = torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 5))
        return x, boxes, p

def det_collate(batch):
    imgs, boxes_list, paths = zip(*batch)
    cls_l, bbox_l, bidx_l = [], [], []
    for i, b in enumerate(boxes_list):
        if b.shape[0] > 0:
            cls_l.append(b[:, 0:1]); bbox_l.append(b[:, 1:5])
            bidx_l.append(torch.full((b.shape[0],), i))
    targets = {
        'cls':       torch.cat(cls_l, 0)  if cls_l  else torch.zeros((0, 1)),
        'bboxes':    torch.cat(bbox_l, 0) if bbox_l else torch.zeros((0, 4)),
        'batch_idx': torch.cat(bidx_l, 0) if bidx_l else torch.zeros((0,)),
    }
    return torch.stack(imgs), targets, paths

det_test_ds = DFireDetDataset(bld.DFIRE_ROOT, split='test', size=384)
det_test_loader = DataLoader(det_test_ds, batch_size=8, shuffle=False, collate_fn=det_collate)

In [ ]:
import shutil

def save_as_dfire_split(x_batch, paths, save_dir):
    os.makedirs(f"{save_dir}/images", exist_ok=True)
    os.makedirs(f"{save_dir}/labels", exist_ok=True)
    for img, p in zip(x_batch, paths):
        name = os.path.basename(p).rsplit('.', 1)[0]
        T.ToPILImage()(img.cpu()).save(f"{save_dir}/images/{name}.jpg")
        lp = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        if os.path.exists(lp):
            shutil.copy(lp, f"{save_dir}/labels/{name}.txt")

def eval_map(save_dir, img_size=384):
    yaml_path = f"{save_dir}/tmp.yaml"
    with open(yaml_path, 'w') as f:
        yaml.dump({'path': save_dir, 'train': 'images', 'val': 'images',
                   'names': {0: 'fire', 1: 'smoke'}}, f)
    m = detector.val(data=yaml_path, imgsz=img_size, verbose=False)
    return m.box.map50, m.box.map

In [ ]:
from ultralytics.cfg import get_cfg, DEFAULT_CFG

attack_yolo.model.args = get_cfg(DEFAULT_CFG)   # 拿到官方默认的完整超参（含box/cls/dfl等）
criterion = v8DetectionLoss(attack_yolo.model)  # 重新初始化

print(criterion.hyp.box, criterion.hyp.cls, criterion.hyp.dfl)  # 应该输出 7.5 0.5 1.5

In [ ]:
"""
Cell 7 — 完整测试集评测（断点续跑版）
依赖：ae, diff_model, betas, flip_prob, attack_yolo, criterion, detector,
      pgd_attack, DFireDetDataset, det_collate, save_as_dfire_split, eval_map
      (这些应该都已经在你的Colab session里定义好了，来自 Cell1~6)
"""

import os
import yaml
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

# ---------------- 配置 ----------------
N_EVAL   = None          # None = 跑完整test set；先设成比如300做个中等规模验证也可以
T_STAR   = 40             # 先用单一t*跑全量，确认流程稳定后再考虑扫描多个t*
BATCH    = 8
OUT_ROOT = '/content/drive/MyDrive/dfire_adv_eval'   # 存Drive，断连不丢

for split in ['clean', 'adv', 'purified']:
    os.makedirs(f'{OUT_ROOT}/{split}/images', exist_ok=True)
    os.makedirs(f'{OUT_ROOT}/{split}/labels', exist_ok=True)

progress_path = f'{OUT_ROOT}/done.txt'
done = set()
if os.path.exists(progress_path):
    done = set(open(progress_path).read().split())
print(f"已完成 {len(done)} 张图（断点续跑会自动跳过它们）")

det_ds = DFireDetDataset(bld.DFIRE_ROOT, split='test', size=384)
if N_EVAL:
    det_ds.img_paths = det_ds.img_paths[:N_EVAL]
det_loader = DataLoader(det_ds, batch_size=BATCH, shuffle=False, collate_fn=det_collate)
print(f"本次评测图片总数: {len(det_ds)}")

# ---------------- 主循环：逐batch攻击 + 净化 + 存盘 ----------------
progress_f = open(progress_path, 'a')

for x, targets, paths in tqdm(det_loader, desc="attack+purify"):
    names = [os.path.basename(p) for p in paths]
    if all(n in done for n in names):
        continue   # 这个batch全部处理过，跳过

    x = x.to(bld.DEVICE)

    # 有些图是background（没有框），PGD无法针对空targets求loss，直接跳过攻击，
    # 但仍然要保留这些图参与最终mAP统计（它们贡献recall/precision里的负样本部分）
    if targets['bboxes'].shape[0] == 0:
        x_adv = x.clone()
    else:
        targets_gpu = {k: v.to(bld.DEVICE) for k, v in targets.items()}
        x_adv = pgd_attack(attack_yolo.model, criterion, x, targets_gpu)

    # resize到AE训练分辨率(128)做净化，再resize回384给检测器用
    x_adv_128 = F.interpolate(x_adv, size=128, mode='bilinear', align_corners=False)
    with torch.no_grad():
        z_adv = ae.encode_binary(x_adv_128)
        t_vec = torch.full((z_adv.shape[0],), T_STAR - 1,
                            device=bld.DEVICE, dtype=torch.long)
        z_noised = bld.q_sample(z_adv, t_vec, flip_prob)
        z_rec    = bld.reverse(diff_model, z_noised, T_STAR, betas, flip_prob)
        x_pur_128 = ae.decode(z_rec.float())
    x_pur = F.interpolate(x_pur_128, size=384, mode='bilinear', align_corners=False)

    save_as_dfire_split(x,       paths, f'{OUT_ROOT}/clean')
    save_as_dfire_split(x_adv,   paths, f'{OUT_ROOT}/adv')
    save_as_dfire_split(x_pur,   paths, f'{OUT_ROOT}/purified')

    for n in names:
        progress_f.write(n + '\n')
    progress_f.flush()
    done.update(names)

progress_f.close()
print(f"\n全部处理完成，共 {len(done)} 张图已存入 {OUT_ROOT}")


# ---------------- 全量mAP对比 ----------------
print("\n=== 完整测试集 mAP 对比 ===")
results = {}
for name in ['clean', 'adv', 'purified']:
    map50, map5095 = eval_map(f'{OUT_ROOT}/{name}', img_size=384)
    results[name] = (map50, map5095)
    print(f"{name:10s}  mAP50={map50:.4f}  mAP50-95={map5095:.4f}")

recovery = (results['purified'][0] - results['adv'][0]) / max(results['clean'][0] - results['adv'][0], 1e-8)
print(f"\n净化恢复比例 (purified-adv)/(clean-adv) = {recovery:.1%}")

In [ ]:
"""
Cell 7.5 — 诊断实验：只过 AE (encode+decode)，不做PGD攻击、不做扩散净化
目的：拆分 "AE压缩瓶颈" 和 "扩散净化过程" 各自对mAP下降的贡献
依赖：ae, det_loader (来自Cell7), OUT_ROOT, save_as_dfire_split, eval_map
"""

import os
import torch
import torch.nn.functional as F
from tqdm import tqdm

AE_ONLY_DIR = f'{OUT_ROOT}/ae_only'
os.makedirs(f'{AE_ONLY_DIR}/images', exist_ok=True)
os.makedirs(f'{AE_ONLY_DIR}/labels', exist_ok=True)

progress_path_ae = f'{OUT_ROOT}/done_ae_only.txt'
done_ae = set()
if os.path.exists(progress_path_ae):
    done_ae = set(open(progress_path_ae).read().split())
print(f"ae_only 已完成 {len(done_ae)} 张图")

progress_f_ae = open(progress_path_ae, 'a')

for x, targets, paths in tqdm(det_loader, desc="ae_only (no attack, no purify)"):
    names = [os.path.basename(p) for p in paths]
    if all(n in done_ae for n in names):
        continue

    x = x.to(bld.DEVICE)   # ← 关键：新循环里正常拿到当前batch并搬到GPU，不会用到过期变量

    with torch.no_grad():
        x_128 = F.interpolate(x, size=128, mode='bilinear', align_corners=False)
        z_clean = ae.encode_binary(x_128)
        x_ae_128 = ae.decode(z_clean.float())
    x_ae = F.interpolate(x_ae_128, size=384, mode='bilinear', align_corners=False)

    save_as_dfire_split(x_ae, paths, AE_ONLY_DIR)

    for n in names:
        progress_f_ae.write(n + '\n')
    progress_f_ae.flush()
    done_ae.update(names)

progress_f_ae.close()
print(f"\nae_only 全部处理完成，共 {len(done_ae)} 张图")

# ---------------- mAP ----------------
map50, map5095 = eval_map(AE_ONLY_DIR, img_size=384)
print(f"\nae_only    mAP50={map50:.4f}  mAP50-95={map5095:.4f}")
print("（对照：clean=0.5566, adv=0.0243→已更新为0.0759, purified=0.0568，具体以你实际这轮的结果为准）")

In [ ]:
"""
快速验证：task-aware AE 是否值得投入完整训练
改动：Stage 1 训练loop里加一项"检测backbone特征loss"，只跑5个epoch（而不是20），
      跑完立刻在一个小子集上测 ae_only 的 mAP，看有没有明显回升。
依赖：ae(或重新构造一个新的BinaryAutoEncoder实例)、attack_yolo(未被fuse过的YOLOv8实例，
      来自之前PGD攻击部分)、bld模块、det_loader
"""

import torch
import torch.nn.functional as F
from tqdm import tqdm

# ---------------- 1. 注册hook，从YOLOv8 backbone中间层拿特征 ----------------
# attack_yolo.model 是ultralytics DetectionModel，model.model是Sequential of layers
# 先打印一下确认层结构，选一个backbone中段的层（不要选太深的检测头部分）
print(attack_yolo.model.model)   # 跑一次，确认要hook哪一层，默认先试 layer index 6

FEAT_LAYER_IDX = 6   # 根据上面打印结果调整；建议选backbone中段（P3/P4附近）
_feat_store = {}

def _hook(module, inp, out):
    _feat_store['feat'] = out

handle = attack_yolo.model.model[FEAT_LAYER_IDX].register_forward_hook(_hook)

for p in attack_yolo.model.parameters():
    p.requires_grad_(False)   # 检测器权重全程冻结，只当特征提取器用
attack_yolo.model.eval()

def get_det_feat(x_384):
    """x_384: (B,3,384,384) in [0,1] -> backbone中间层特征"""
    attack_yolo.model(x_384)
    return _feat_store['feat']


# ---------------- 2. Task-aware AE 训练（只跑几个epoch做快速验证） ----------------
LAMBDA_DET = 1.0     # 检测loss权重，先给个粗略值，后面可以再调
QUICK_EPOCHS = 5

def train_autoencoder_taskaware(ae, train_loader, epochs=QUICK_EPOCHS, lambda_det=LAMBDA_DET):
    opt = torch.optim.Adam(ae.parameters(), lr=bld.LR_AE)

    for epoch in range(epochs):
        ae.train()
        total, total_mse, total_det, n = 0., 0., 0., 0

        for x, _ in tqdm(train_loader, desc=f"quick AE epoch {epoch+1}", leave=False):
            x = x.to(bld.DEVICE)                                   # (B,3,128,128)
            x_rec, z = ae(x)

            mse = F.mse_loss(x_rec, x)
            z_soft = z.float()
            bin_reg = -(z_soft * torch.log(z_soft + 1e-6) +
                        (1 - z_soft) * torch.log(1 - z_soft + 1e-6)).mean()

            # 检测特征loss：把重建图和原图都resize到384，喂进冻结的YOLO backbone对比中间特征
            x_384      = F.interpolate(x,     size=384, mode='bilinear', align_corners=False)
            x_rec_384  = F.interpolate(x_rec, size=384, mode='bilinear', align_corners=False)
            with torch.no_grad():
                feat_real = get_det_feat(x_384)
            feat_rec = get_det_feat(x_rec_384)
            det_loss = F.mse_loss(feat_rec, feat_real.detach())

            loss = mse + 0.01 * bin_reg + lambda_det * det_loss

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ae.parameters(), 1.0)
            opt.step()

            bs = x.shape[0]
            total     += loss.item() * bs
            total_mse += mse.item() * bs
            total_det += det_loss.item() * bs
            n += bs

        print(f"  epoch {epoch+1}: loss={total/n:.4f}  mse={total_mse/n:.4f}  det={total_det/n:.4f}")

    return ae

# 用一个全新的AE实例训练（不覆盖你现有的ae/CKPT_AE，避免破坏已有checkpoint）
ae_taskaware = bld.BinaryAutoEncoder().to(bld.DEVICE)
ae_taskaware = train_autoencoder_taskaware(ae_taskaware, bld.get_loaders()[0])

handle.remove()   # 训练完记得摘掉hook，避免影响后续attack_yolo的正常调用
torch.save(ae_taskaware.state_dict(), '/content/drive/MyDrive/bld_purify_checkpoints/ae_taskaware_quick.pt')
print("quick task-aware AE 已存盘")


# ---------------- 3. 小子集上快速测 ae_only 的 mAP ----------------
import os

QUICK_DIR = '/content/drive/MyDrive/dfire_adv_eval/ae_only_taskaware_quick'
os.makedirs(f'{QUICK_DIR}/images', exist_ok=True)
os.makedirs(f'{QUICK_DIR}/labels', exist_ok=True)

ae_taskaware.eval()
n_done = 0
for x, targets, paths in det_loader:          # 复用之前定义好的det_loader
    x = x.to(bld.DEVICE)
    with torch.no_grad():
        x_128 = F.interpolate(x, size=128, mode='bilinear', align_corners=False)
        x_rec_128, _ = ae_taskaware(x_128)
    x_rec_384 = F.interpolate(x_rec_128, size=384, mode='bilinear', align_corners=False)
    save_as_dfire_split(x_rec_384, paths, QUICK_DIR)
    n_done += len(paths)
    if n_done >= 300:      # 只测300张做快速验证，不用等全量
        break

map50, map5095 = eval_map(QUICK_DIR, img_size=384)
print(f"\ntask-aware ae_only (quick, {n_done}张)  mAP50={map50:.4f}  mAP50-95={map5095:.4f}")
print("对照：原始ae_only(全量4306张) mAP50=0.0823 —— 如果这里明显更高，说明方向有效，值得投入完整训练")

In [ ]:
import os
from tqdm import tqdm
QUICK_DIR = '/content/drive/MyDrive/dfire_adv_eval/ae_only_taskaware_quick'
os.makedirs(f'{QUICK_DIR}/images', exist_ok=True)
os.makedirs(f'{QUICK_DIR}/labels', exist_ok=True)

ae_taskaware.eval()
n_done = 0
for x, targets, paths in tqdm(det_loader, desc="ae_only taskaware full eval"):
    x = x.to(bld.DEVICE)
    with torch.no_grad():
        x_128 = F.interpolate(x, size=128, mode='bilinear', align_corners=False)
        x_rec_128, _ = ae_taskaware(x_128)
    x_rec_384 = F.interpolate(x_rec_128, size=384, mode='bilinear', align_corners=False)
    save_as_dfire_split(x_rec_384, paths, QUICK_DIR)
    n_done += len(paths)

map50, map5095 = eval_map(QUICK_DIR, img_size=384)
print(f"\ntask-aware ae_only (全量{n_done}张)  mAP50={map50:.4f}  mAP50-95={map5095:.4f}")
print("对照：原始ae_only(全量4306张) mAP50=0.0823")

In [ ]:
"""
5-epoch 纯MSE baseline（不加det_loss），跟5-epoch task-aware版本做公平对照
关键：临时替换 bld.CKPT_AE 和 bld.EPOCHS_AE，跑完立刻恢复原值，
      避免覆盖你现有的20epoch checkpoint (bld_autoencoder.pt)
"""

import torch, os
import torch.nn.functional as F
from tqdm import tqdm

# ---------------- 1. 临时改路径和epoch数，跑完自动恢复 ----------------
_orig_ckpt_ae   = bld.CKPT_AE
_orig_epochs_ae = bld.EPOCHS_AE

bld.CKPT_AE   = '/content/drive/MyDrive/bld_purify_checkpoints/ae_baseline_5ep.pt'
bld.EPOCHS_AE = 5

ae_baseline_5ep = bld.BinaryAutoEncoder().to(bld.DEVICE)
train_loader, test_loader = bld.get_loaders()
ae_baseline_5ep = bld.train_autoencoder(ae_baseline_5ep, train_loader, test_loader)

# 恢复原值，避免影响这个session里后续任何其他用到bld.CKPT_AE/EPOCHS_AE的代码
bld.CKPT_AE   = _orig_ckpt_ae
bld.EPOCHS_AE = _orig_epochs_ae
print(f"已恢复 bld.CKPT_AE = {bld.CKPT_AE}, bld.EPOCHS_AE = {bld.EPOCHS_AE}")

# ---------------- 2. 全量跑 ae_only 评测，对比公平（同样5epoch，只是没有det_loss）----------------
BASELINE_DIR = '/content/drive/MyDrive/dfire_adv_eval/ae_only_baseline_5ep'
os.makedirs(f'{BASELINE_DIR}/images', exist_ok=True)
os.makedirs(f'{BASELINE_DIR}/labels', exist_ok=True)

ae_baseline_5ep.eval()
n_done = 0
for x, targets, paths in tqdm(det_loader, desc="ae_only baseline_5ep full eval"):
    x = x.to(bld.DEVICE)
    with torch.no_grad():
        x_128 = F.interpolate(x, size=128, mode='bilinear', align_corners=False)
        x_rec_128, _ = ae_baseline_5ep(x_128)
    x_rec_384 = F.interpolate(x_rec_128, size=384, mode='bilinear', align_corners=False)
    save_as_dfire_split(x_rec_384, paths, BASELINE_DIR)
    n_done += len(paths)

map50, map5095 = eval_map(BASELINE_DIR, img_size=384)
print(f"\nbaseline_5ep (纯MSE, 全量{n_done}张)  mAP50={map50:.4f}  mAP50-95={map5095:.4f}")
print("对照：")
print("  task-aware 5ep  mAP50=0.0190")
print("  原始AE    20ep  mAP50=0.0823")

In [ ]:
"""
从现有20epoch AE checkpoint热启动，用小权重det_loss微调，避免破坏已学好的像素重建能力
依赖：attack_yolo(未fuse的YOLOv8实例)、bld模块、det_loader、save_as_dfire_split、eval_map
"""

import torch
import torch.nn.functional as F
from tqdm import tqdm
import os

# ---------------- 1. 重新挂hook（如果是新session或者之前已经remove过）----------------
FEAT_LAYER_IDX = 6
_feat_store = {}

def _hook(module, inp, out):
    _feat_store['feat'] = out

handle = attack_yolo.model.model[FEAT_LAYER_IDX].register_forward_hook(_hook)
for p in attack_yolo.model.parameters():
    p.requires_grad_(False)
attack_yolo.model.eval()

def get_det_feat(x_384):
    attack_yolo.model(x_384)
    return _feat_store['feat']

# ---------------- 2. 微调函数：小学习率、小lambda_det，从热启动权重继续训 ----------------
def finetune_taskaware(ae, train_loader, epochs=5, lambda_det=0.1, lr=1e-4):
    opt = torch.optim.Adam(ae.parameters(), lr=lr)   # 微调用更小的lr，避免破坏已学好的重建能力

    for epoch in range(epochs):
        ae.train()
        total_mse, total_det, n = 0., 0., 0

        for x, _ in tqdm(train_loader, desc=f"finetune epoch {epoch+1}", leave=False):
            x = x.to(bld.DEVICE)
            x_rec, z = ae(x)

            mse = F.mse_loss(x_rec, x)
            z_soft = z.float()
            bin_reg = -(z_soft * torch.log(z_soft + 1e-6) +
                        (1 - z_soft) * torch.log(1 - z_soft + 1e-6)).mean()

            x_384     = F.interpolate(x,     size=384, mode='bilinear', align_corners=False)
            x_rec_384 = F.interpolate(x_rec, size=384, mode='bilinear', align_corners=False)
            with torch.no_grad():
                feat_real = get_det_feat(x_384)
            feat_rec = get_det_feat(x_rec_384)
            det_loss = F.mse_loss(feat_rec, feat_real.detach())

            loss = mse + 0.01 * bin_reg + lambda_det * det_loss

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ae.parameters(), 1.0)
            opt.step()

            bs = x.shape[0]
            total_mse += mse.item() * bs
            total_det += det_loss.item() * bs
            n += bs

        print(f"  epoch {epoch+1}: mse={total_mse/n:.4f}  det={total_det/n:.4f}")

    return ae

# ---------------- 3. 从现有20epoch checkpoint热启动 ----------------
ae_taskaware_ft = bld.BinaryAutoEncoder().to(bld.DEVICE)
ae_taskaware_ft.load_state_dict(torch.load(bld.CKPT_AE, map_location=bld.DEVICE))  # 热启动
print(f"已从 {bld.CKPT_AE} 热启动")

train_loader, _ = bld.get_loaders()
ae_taskaware_ft = finetune_taskaware(ae_taskaware_ft, train_loader, epochs=5, lambda_det=0.1)

handle.remove()
torch.save(ae_taskaware_ft.state_dict(),
           '/content/drive/MyDrive/bld_purify_checkpoints/ae_taskaware_finetuned.pt')
print("微调后的AE已存盘")

# ---------------- 4. 全量评测，对比基线0.0823 ----------------
FT_DIR = '/content/drive/MyDrive/dfire_adv_eval/ae_only_taskaware_finetuned'
os.makedirs(f'{FT_DIR}/images', exist_ok=True)
os.makedirs(f'{FT_DIR}/labels', exist_ok=True)

ae_taskaware_ft.eval()
n_done = 0
for x, targets, paths in tqdm(det_loader, desc="ae_only taskaware_finetuned full eval"):
    x = x.to(bld.DEVICE)
    with torch.no_grad():
        x_128 = F.interpolate(x, size=128, mode='bilinear', align_corners=False)
        x_rec_128, _ = ae_taskaware_ft(x_128)
    x_rec_384 = F.interpolate(x_rec_128, size=384, mode='bilinear', align_corners=False)
    save_as_dfire_split(x_rec_384, paths, FT_DIR)
    n_done += len(paths)

map50, map5095 = eval_map(FT_DIR, img_size=384)
print(f"\ntask-aware finetuned (全量{n_done}张)  mAP50={map50:.4f}  mAP50-95={map5095:.4f}")
print("对照：原始AE 20ep (纯MSE)  mAP50=0.0823")

In [ ]:
"""
Stage2 重训（热启动版）：用 ae_v2(task-aware AE) 产生的binary latent，
继续训练现有的 diff_model，而不是从零开始
依赖：ae_v2, diff_model, betas, flip_prob, bld模块
"""

import torch

# ---------------- 1. 热启动：diff_model已经在session里了，直接继续训 ----------------
# 如果是新session，先重新加载一次：
diff_model = bld.ReverseModel(D=ae_attn.D).to(bld.DEVICE)
diff_model.load_state_dict(torch.load('/content/drive/MyDrive/bld_purify_checkpoints/bld_diffusion.pt', map_location=bld.DEVICE))
# 1. 先加载 ae_v2 (task-aware微调后的AE)
ae_attn = bld.BinaryAutoEncoder().to(bld.DEVICE)
ae_attn.load_state_dict(
    torch.load('/content/drive/MyDrive/bld_purify_checkpoints/ae_taskaware_finetuned.pt',
               map_location=bld.DEVICE)
)
ae_attn.eval()
print("ae_v2 已加载")

# 2. 再加载 diff_model（热启动，从原始的bld_diffusion.pt继续训）
diff_model_v3 = bld.ReverseModel(D=ae_attn.D).to(bld.DEVICE)
diff_model_v3.load_state_dict(
    torch.load('/content/drive/MyDrive/bld_purify_checkpoints/bld_diffusion_v2.pt', map_location=bld.DEVICE)
)
print("diff_model 已加载（热启动用）")

# 3. betas, flip_prob 也需要重新生成
betas, flip_prob = bld.get_schedule()
train_loader, _ = bld.get_loaders()

# ---------------- 2. 关键：把ae换成ae_v2，其余逻辑复用bld.train_diffusion ----------------
EPOCHS_RETRAIN = 10   # 热启动，不用像第一次训20epoch那么久，先跑10epoch看收敛趋势

diff_model_v3 = bld.train_diffusion(diff_model_v3, ae_attn, train_loader, betas, flip_prob)   # 如果函数不接受epochs参数，见下方说明

torch.save(diff_model_v3.state_dict(),
           '/content/drive/MyDrive/bld_purify_checkpoints/bld_diffusion_v2.pt')
print("diff_model_v3 已存盘")

In [ ]:
import os
import torch
import torch.nn.functional as F
from tqdm import tqdm

T_STAR   = 40
OUT_ROOT_V4 = '/content/drive/MyDrive/dfire_adv_eval_v4'   # 新目录，第四轮结果
for split in ['clean', 'adv', 'purified']:
    os.makedirs(f'{OUT_ROOT_V4}/{split}/images', exist_ok=True)
    os.makedirs(f'{OUT_ROOT_V4}/{split}/labels', exist_ok=True)

progress_path = f'{OUT_ROOT_V4}/done.txt'
done = set(open(progress_path).read().split()) if os.path.exists(progress_path) else set()
print(f"已完成 {len(done)} 张图")

progress_f = open(progress_path, 'a')

for x, targets, paths in tqdm(det_loader, desc="v4 attack+purify"):
    names = [os.path.basename(p) for p in paths]
    if all(n in done for n in names):
        continue

    x = x.to(bld.DEVICE)

    if targets['bboxes'].shape[0] == 0:
        x_adv = x.clone()
    else:
        targets_gpu = {k: v.to(bld.DEVICE) for k, v in targets.items()}
        x_adv = pgd_attack(attack_yolo.model, criterion, x, targets_gpu)

    x_adv_128 = F.interpolate(x_adv, size=128, mode='bilinear', align_corners=False)
    with torch.no_grad():
        z_adv = ae_attn.encode_binary(x_adv_128)                          # ← 换成ae_attn
        t_vec = torch.full((z_adv.shape[0],), T_STAR - 1,
                            device=bld.DEVICE, dtype=torch.long)
        z_noised = bld.q_sample(z_adv, t_vec, flip_prob)
        z_rec    = bld.reverse(diff_model_v3, z_noised, T_STAR, betas, flip_prob)  # ← 换成diff_model_v3
        x_pur_128 = ae_attn.decode(z_rec.float())                         # ← 换成ae_attn
    x_pur = F.interpolate(x_pur_128, size=384, mode='bilinear', align_corners=False)

    save_as_dfire_split(x,      paths, f'{OUT_ROOT_V4}/clean')
    save_as_dfire_split(x_adv,  paths, f'{OUT_ROOT_V4}/adv')
    save_as_dfire_split(x_pur,  paths, f'{OUT_ROOT_V4}/purified')

    for n in names:
        progress_f.write(n + '\n')
    progress_f.flush()
    done.update(names)

progress_f.close()
print(f"\n全部完成，共 {len(done)} 张图")

print("\n=== v4 (attn AE + 重训diffusion) 完整测试集 mAP 对比 ===")
results = {}
for name in ['clean', 'adv', 'purified']:
    map50, map5095 = eval_map(f'{OUT_ROOT_V4}/{name}', img_size=384)
    results[name] = (map50, map5095)
    print(f"{name:10s}  mAP50={map50:.4f}  mAP50-95={map5095:.4f}")

recovery = (results['purified'][0] - results['adv'][0]) / max(results['clean'][0] - results['adv'][0], 1e-8)
print(f"\n净化恢复比例 (purified-adv)/(clean-adv) = {recovery:.1%}")

print("\n=== 三轮对照 ===")
print("v1 (原始AE+原始diff)     purified=0.0568  恢复比例=-4.0%")
print("v2 (det_loss AE+原始diff) purified=0.0997  恢复比例=+5.3%")
print("v3 (det_loss AE+重训diff) purified=0.1118  恢复比例=+7.6%")

In [ ]:
"""
ae_large 两阶段训练：
  Phase 1 — 冻结 g_a/g_s（CompressAI预训练backbone），只训练新增的二值化头（to_binary/from_binary）
  Phase 2 — 解冻全部，联合微调（小学习率，避免破坏预训练权重）
依赖：ae_large, bld模块, attack_yolo(如果hook还没挂需要重新挂)
"""

import torch
import torch.nn.functional as F
from tqdm import tqdm

# ---------------- 0. 确认/重新挂检测特征hook（如果新session要重新跑这几行） ----------------
if 'get_det_feat' not in dir():
    FEAT_LAYER_IDX = 6
    _feat_store = {}

    def _hook(module, inp, out):
        _feat_store['feat'] = out

    handle = attack_yolo.model.model[FEAT_LAYER_IDX].register_forward_hook(_hook)
    for p in attack_yolo.model.parameters():
        p.requires_grad_(False)
    attack_yolo.model.eval()

    def get_det_feat(x_384):
        attack_yolo.model(x_384)
        return _feat_store['feat']
    print("已重新挂载检测特征hook")


# ---------------- 1. 训练函数（跟finetune_taskaware逻辑一致，只是ae_large没有y参数） ----------------
def train_large_ae(ae, train_loader, epochs, lambda_det=0.1, lr=1e-4):
    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, ae.parameters()), lr=lr)
    for epoch in range(epochs):
        ae.train()
        total_mse, total_det, n = 0., 0., 0
        for x, _ in tqdm(train_loader, desc=f"epoch {epoch+1}", leave=False):
            x = x.to(bld.DEVICE)
            x_rec, z = ae(x)

            mse = F.mse_loss(x_rec, x)
            z_soft = z.float()
            bin_reg = -(z_soft * torch.log(z_soft + 1e-6) +
                        (1 - z_soft) * torch.log(1 - z_soft + 1e-6)).mean()

            x_384     = F.interpolate(x,     size=384, mode='bilinear', align_corners=False)
            x_rec_384 = F.interpolate(x_rec, size=384, mode='bilinear', align_corners=False)
            with torch.no_grad():
                feat_real = get_det_feat(x_384)
            feat_rec = get_det_feat(x_rec_384)
            det_loss = F.mse_loss(feat_rec, feat_real.detach())

            loss = mse + 0.01 * bin_reg + lambda_det * det_loss
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ae.parameters(), 1.0)
            opt.step()

            bs = x.shape[0]
            total_mse += mse.item() * bs
            total_det += det_loss.item() * bs
            n += bs
        print(f"  epoch {epoch+1}: mse={total_mse/n:.4f}  det={total_det/n:.4f}")
    return ae


# ---------------- 2. Phase 1：冻结backbone，只训二值化头 ----------------
train_loader, _ = bld.get_loaders()

for p in ae_large.parameters():
    p.requires_grad_(False)
for p in ae_large.encoder.to_binary.parameters():
    p.requires_grad_(True)
for p in ae_large.decoder.from_binary.parameters():
    p.requires_grad_(True)

n_trainable = sum(p.numel() for p in ae_large.parameters() if p.requires_grad)
print(f"Phase 1: 只训二值化头，可训练参数量 = {n_trainable}")

ae_large = train_large_ae(ae_large, train_loader, epochs=5, lambda_det=0.1, lr=3e-4)

# ---------------- 3. Phase 2：解冻g_a/g_s，联合微调（小学习率） ----------------
for p in ae_large.parameters():
    p.requires_grad_(True)

n_trainable = sum(p.numel() for p in ae_large.parameters() if p.requires_grad)
print(f"Phase 2: 解冻全部联合微调，可训练参数量 = {n_trainable}")

ae_large = train_large_ae(ae_large, train_loader, epochs=10, lambda_det=0.1, lr=1e-5)
# 注意lr比之前所有实验都小(1e-5)：这次的backbone是在大规模数据集上预训练的，
# D-Fire数据量相对小，用更保守的学习率避免灾难性遗忘预训练学到的通用特征

torch.save(ae_large.state_dict(), '/content/drive/MyDrive/bld_purify_checkpoints/ae_large_cheng2020.pt')
print("ae_large 两阶段训练完成并已存盘")

In [ ]:
import os
LARGE_DIR = '/content/drive/MyDrive/dfire_adv_eval/ae_only_large'
os.makedirs(f'{LARGE_DIR}/images', exist_ok=True)
os.makedirs(f'{LARGE_DIR}/labels', exist_ok=True)

ae_large.eval()
for x, targets, paths in tqdm(det_loader, desc="ae_only large full eval"):
    x = x.to(bld.DEVICE)
    with torch.no_grad():
        x_128 = F.interpolate(x, size=128, mode='bilinear', align_corners=False)
        x_rec_128, _ = ae_large(x_128)
    x_rec_384 = F.interpolate(x_rec_128, size=384, mode='bilinear', align_corners=False)
    save_as_dfire_split(x_rec_384, paths, LARGE_DIR)

map50, map5095 = eval_map(LARGE_DIR, img_size=384)
print(f"\nae_only (CompressAI大backbone)  mAP50={map50:.4f}  mAP50-95={map5095:.4f}")
print("对照：")
print("  attn(zero-init)      mAP50=0.1814")
print("  task-aware(无attn)   mAP50=0.1526")
print("  原始AE               mAP50=0.0823")

In [ ]:
"""
在现有 ae_attn (32x32处zero-init self-attention) 基础上，加入 label-conditioned FiLM，
实现类似 PuVAE 的条件净化：D-Fire只有fire/smoke两类，4种multi-hot组合(00,01,10,11)，
推理时穷举4个分支，用YOLOv8置信度选最优分支。
依赖：bld模块, EncoderAttn/DecoderAttn/BinaryAutoEncoderAttn (来自 ae_add_attention_32x32.py)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import glob, os
from PIL import Image
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader


# ---------------- 1. FiLM条件调制层 ----------------
class LabelConditioner(nn.Module):
    def __init__(self, label_dim=2, feature_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(label_dim, feature_dim),
            nn.SiLU(),
            nn.Linear(feature_dim, feature_dim * 2),
        )
        # zero-init最后一层：起点 gamma=0, beta=0 -> feat*(1+0)+0=feat，不扰动已有权重
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)

    def forward(self, feat, y_multihot):
        gamma_beta = self.mlp(y_multihot)
        gamma, beta = gamma_beta.chunk(2, dim=-1)
        gamma = gamma.unsqueeze(-1).unsqueeze(-1)
        beta = beta.unsqueeze(-1).unsqueeze(-1)
        return feat * (1 + gamma) + beta


# ---------------- 2. 包一层FiLM在原有的zero-init AttnBlock外面 ----------------
class ConditionedAttnBlock(nn.Module):
    def __init__(self, channels, label_dim=2):
        super().__init__()
        self.attn = bld.AttnBlock(channels)   # 结构和之前插入的一致，权重稍后手动搬运
        self.film = LabelConditioner(label_dim, channels)

    def forward(self, x, y):
        x = self.film(x, y)
        return self.attn(x)


# ---------------- 3. 改造Encoder/Decoder：conv_layers里第9(enc)/7(dec)层换成条件版本 ----------------
class EncoderAttnCond(nn.Module):
    def __init__(self, base_encoder_attn):
        super().__init__()
        # 复用EncoderAttn已经建好的层，只把index 9的AttnBlock换成ConditionedAttnBlock
        old_layers = list(base_encoder_attn.conv_layers)
        self.pre  = nn.Sequential(*old_layers[:9])
        self.cond_attn = ConditionedAttnBlock(old_layers[9].proj.out_channels)
        self.post = nn.Sequential(*old_layers[10:])

    def forward(self, x, y, hard=True):
        h = self.pre(x)
        h = self.cond_attn(h, y)
        logits = self.post(h)
        if not hard:
            return torch.sigmoid(logits)
        z_soft = torch.sigmoid(logits)
        z_hard = (z_soft > 0.5).float()
        return z_soft + (z_hard - z_soft).detach()


class DecoderAttnCond(nn.Module):
    def __init__(self, base_decoder_attn):
        super().__init__()
        old_layers = list(base_decoder_attn.conv_layers)
        self.pre  = nn.Sequential(*old_layers[:7])
        self.cond_attn = ConditionedAttnBlock(old_layers[7].proj.out_channels)
        self.post = nn.Sequential(*old_layers[8:])

    def forward(self, z, y):
        h = self.pre(z)
        h = self.cond_attn(h, y)
        return self.post(h)


class BinaryAutoEncoderAttnCond(nn.Module):
    def __init__(self, base_ae_attn):
        super().__init__()
        self.encoder = EncoderAttnCond(base_ae_attn.encoder)
        self.decoder = DecoderAttnCond(base_ae_attn.decoder)
        self.latent_h = base_ae_attn.latent_h
        self.latent_w = base_ae_attn.latent_w
        self.D = base_ae_attn.D
        # 把原本zero-init AttnBlock的权重搬进新的cond_attn.attn子模块
        self.encoder.cond_attn.attn.load_state_dict(base_ae_attn.encoder.conv_layers[9].state_dict())
        self.decoder.cond_attn.attn.load_state_dict(base_ae_attn.decoder.conv_layers[7].state_dict())
        print("已从ae_attn搬运attn权重，FiLM层保持zero-init（起点=无条件版本完全一致）")

    def encode_binary(self, x, y):
        with torch.no_grad():
            z = self.encoder(x, y, hard=True)
        return z.long().view(z.shape[0], -1)

    def decode(self, z_flat, y):
        B = z_flat.shape[0]
        z = z_flat.view(B, bld.LATENT_CHANNELS, self.latent_h, self.latent_w)
        return self.decoder(z, y)

    def forward(self, x, y):
        z = self.encoder(x, y, hard=True)
        z_flat = z.view(z.shape[0], -1)
        x_rec = self.decode(z_flat, y)
        return x_rec, z


# ---------------- 4. D-Fire multi-hot标签数据集（AE训练用，128分辨率） ----------------
class DFireMultihotDataset(Dataset):
    def __init__(self, root, split='train', size=128):
        self.img_paths = sorted(glob.glob(f'{root}/{split}/images/*.jpg'))
        self.label_dir = f'{root}/{split}/labels'
        self.tf = T.Compose([T.Resize((size, size)), T.ToTensor()])

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        p = self.img_paths[idx]
        x = self.tf(Image.open(p).convert('RGB'))
        lp = os.path.join(self.label_dir, os.path.basename(p).rsplit('.', 1)[0] + '.txt')
        y = torch.zeros(2)
        if os.path.exists(lp):
            with open(lp) as f:
                for line in f:
                    parts = line.split()
                    if len(parts) == 5:
                        cls = int(float(parts[0]))
                        if cls in (0, 1):
                            y[cls] = 1.0
        return x, y


# ---------------- 5. 训练：套用你已有的finetune_taskaware逻辑，多传一个y ----------------
def finetune_conditioned(ae_cond, train_loader, epochs=5, lambda_det=0.1, lr=1e-4):
    opt = torch.optim.Adam(ae_cond.parameters(), lr=lr)
    for epoch in range(epochs):
        ae_cond.train()
        total_mse, total_det, n = 0., 0., 0
        for x, y in train_loader:
            x, y = x.to(bld.DEVICE), y.to(bld.DEVICE)
            x_rec, z = ae_cond(x, y)

            mse = F.mse_loss(x_rec, x)
            z_soft = z.float()
            bin_reg = -(z_soft * torch.log(z_soft + 1e-6) +
                        (1 - z_soft) * torch.log(1 - z_soft + 1e-6)).mean()

            x_384     = F.interpolate(x,     size=384, mode='bilinear', align_corners=False)
            x_rec_384 = F.interpolate(x_rec, size=384, mode='bilinear', align_corners=False)
            with torch.no_grad():
                feat_real = get_det_feat(x_384)     # 复用之前挂好的hook
            feat_rec = get_det_feat(x_rec_384)
            det_loss = F.mse_loss(feat_rec, feat_real.detach())

            loss = mse + 0.01 * bin_reg + lambda_det * det_loss
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ae_cond.parameters(), 1.0)
            opt.step()

            bs = x.shape[0]
            total_mse += mse.item() * bs
            total_det += det_loss.item() * bs
            n += bs
        print(f"  epoch {epoch+1}: mse={total_mse/n:.4f}  det={total_det/n:.4f}")
    return ae_cond


# ---------------- 6. 推理：4分支穷举 + YOLOv8置信度选优 ----------------
LABEL_COMBOS = torch.tensor([[0, 0], [1, 0], [0, 1], [1, 1]], dtype=torch.float)

def purify_conditioned(ae_cond, diff_model, x_adv, t_star, betas, flip_prob, yolo_for_scoring):
    """x_adv: (B,3,384,384) in [0,1]，穷举4个label组合，用检测置信度选最优分支"""
    B = x_adv.shape[0]
    x_adv_128 = F.interpolate(x_adv, size=128, mode='bilinear', align_corners=False)

    best_score = torch.full((B,), -1.0, device=x_adv.device)
    best_x_pur = torch.zeros_like(x_adv)

    for combo in LABEL_COMBOS:
        y_batch = combo.unsqueeze(0).expand(B, -1).to(x_adv.device)
        with torch.no_grad():
            z_adv = ae_cond.encode_binary(x_adv_128, y_batch)
            t_vec = torch.full((B,), t_star - 1, device=x_adv.device, dtype=torch.long)
            z_noised = bld.q_sample(z_adv, t_vec, flip_prob)
            z_rec = bld.reverse(diff_model, z_noised, t_star, betas, flip_prob)
            x_pur_128 = ae_cond.decode(z_rec.float(), y_batch)
        x_pur = F.interpolate(x_pur_128, size=384, mode='bilinear', align_corners=False)

        with torch.no_grad():
            preds = yolo_for_scoring.model(x_pur)          # 用未fuse的实例做前向拿置信度
            # preds[0]是检测头原始输出，这里用简化代理：取输出张量绝对值的均值作为置信度近似
            # 更准确的做法是用ultralytics的NMS后处理拿真实置信度，此处先用轻量代理跑通流程
            score = preds[0].abs().mean(dim=[1, 2]) if isinstance(preds, (list, tuple)) else preds.abs().mean(dim=[1, 2])

        improve = score > best_score
        best_score = torch.where(improve, score, best_score)
        best_x_pur[improve] = x_pur[improve]

    return best_x_pur


# ---------------- 7. 执行：构建条件版模型，从ae_attn热启动 ----------------
ae_cond = BinaryAutoEncoderAttnCond(ae_attn).to(bld.DEVICE)

train_ds_multihot = DFireMultihotDataset(bld.DFIRE_ROOT, split='train', size=bld.IMG_SIZE)
train_loader_multihot = DataLoader(train_ds_multihot, batch_size=32, shuffle=True, num_workers=2)

ae_cond = finetune_conditioned(ae_cond, train_loader_multihot, epochs=5, lambda_det=0.1)
torch.save(ae_cond.state_dict(), '/content/drive/MyDrive/bld_purify_checkpoints/ae_cond.pt')
print("ae_cond 训练完成并已存盘")

In [ ]:
train_loader, _ = bld.get_loaders()

ae_attn = finetune_taskaware(ae_attn, train_loader, epochs=5, lambda_det=0.1, lr=2e-4)

torch.save(ae_attn.state_dict(), '/content/drive/MyDrive/bld_purify_checkpoints/ae_attn_taskaware.pt')
print("ae_attn 微调完成并已存盘")

In [ ]:
"""
在AE的Encoder/Decoder里，32x32分辨率处各新增一层self-attention
（原模型只在16x16的bottleneck有attention，这里在更早的分辨率补一层）
从 ae_taskaware_finetuned.pt 手动搬运可复用的权重，只有新插入的attention层是随机初始化
依赖：bld模块 (BASE_CH=32, CH_MULT_ENC=(1,2,4), LATENT_CHANNELS=8, NUM_RES_BLOCKS=2)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

ResnetBlock = bld.ResnetBlock
AttnBlock   = bld.AttnBlock
Downsample  = bld.Downsample
Upsample    = bld.Upsample
Normalize   = bld.Normalize


# ---------------- 1. 新Encoder：在32x32(mul=4阶段的两个ResnetBlock之后、Downsample之前)插入attention ----------------
class EncoderAttn(nn.Module):
    def __init__(self, in_channels=3, base_ch=bld.BASE_CH,
                 num_res_blocks=bld.NUM_RES_BLOCKS,
                 latent_channels=bld.LATENT_CHANNELS,
                 ch_mult=bld.CH_MULT_ENC):
        super().__init__()
        layers = [nn.Conv2d(in_channels, base_ch, 3, padding=1)]      # idx0 stem
        in_ch = base_ch
        for stage_i, mul in enumerate(ch_mult):
            out_ch = base_ch * mul
            for _ in range(num_res_blocks):
                layers.append(ResnetBlock(in_ch, out_ch))
                in_ch = out_ch
            if stage_i == len(ch_mult) - 1:        # 最后一个stage(mul=4)，在Downsample前插入新attention
                layers.append(AttnBlock(in_ch))     # ← 新增：32x32分辨率
            layers.append(Downsample(in_ch))
        layers += [
            ResnetBlock(in_ch),
            AttnBlock(in_ch),                       # 原有的16x16 bottleneck attention
            ResnetBlock(in_ch),
            Normalize(in_ch),
            nn.Conv2d(in_ch, latent_channels, 3, padding=1),
        ]
        self.conv_layers = nn.Sequential(*layers)

    def forward(self, x, hard=True):
        logits = self.conv_layers(x)
        if not hard:
            return torch.sigmoid(logits)
        z_soft = torch.sigmoid(logits)
        z_hard = (z_soft > 0.5).float()
        return z_soft + (z_hard - z_soft).detach()


# ---------------- 2. 新Decoder：对称地在32x32处插入attention ----------------
class DecoderAttn(nn.Module):
    def __init__(self, out_channels=3, base_ch=bld.BASE_CH,
                 num_res_blocks=bld.NUM_RES_BLOCKS,
                 latent_channels=bld.LATENT_CHANNELS,
                 ch_mult=bld.CH_MULT_ENC):
        super().__init__()
        ch_mult_dec = tuple(reversed(ch_mult))
        in_ch = base_ch * ch_mult[-1]
        layers = [
            nn.Conv2d(latent_channels, in_ch, 3, padding=1),
            ResnetBlock(in_ch),
            AttnBlock(in_ch),                       # 原有的16x16 bottleneck attention
            ResnetBlock(in_ch),
        ]
        for stage_i, mul in enumerate(ch_mult_dec):
            out_ch = base_ch * mul
            for _ in range(num_res_blocks):
                layers.append(ResnetBlock(in_ch, out_ch))
                in_ch = out_ch
            layers.append(Upsample(in_ch))
            if stage_i == 0:                        # 对应encoder mul=4阶段，镜像插入
                layers.append(AttnBlock(in_ch))      # ← 新增：32x32分辨率
        layers += [
            Normalize(in_ch),
            nn.Conv2d(in_ch, out_channels, 3, padding=1),
            nn.Sigmoid(),
        ]
        self.conv_layers = nn.Sequential(*layers)

    def forward(self, z):
        return self.conv_layers(z)


class BinaryAutoEncoderAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = EncoderAttn()
        self.decoder = DecoderAttn()
        self.latent_h = bld.IMG_SIZE // (2 ** len(bld.CH_MULT_ENC))
        self.latent_w = self.latent_h
        self.D = bld.LATENT_CHANNELS * self.latent_h * self.latent_w

    def encode_binary(self, x):
        with torch.no_grad():
            z = self.encoder(x, hard=True)
        return z.long().view(z.shape[0], -1)

    def decode(self, z_flat):
        B = z_flat.shape[0]
        z = z_flat.view(B, bld.LATENT_CHANNELS, self.latent_h, self.latent_w)
        return self.decoder(z)

    def forward(self, x):
        z = self.encoder(x, hard=True)
        z_flat = z.view(z.shape[0], -1)
        x_rec = self.decode(z_flat)
        return x_rec, z


# ---------------- 3. 手动搬运权重：老模型的每一层 -> 新模型对应层，新插入的attention跳过（随机初始化） ----------------
def transplant_encoder(old_enc, new_enc):
    # 老encoder的conv_layers索引: 0=stem,1-2=stage1(mul1),3=down,4-5=stage2(mul2),6=down,
    #                            7-8=stage3(mul4),9=down,10=res,11=attn(bottleneck),12=res,13=norm,14=conv_out
    # 新encoder多插入了一层attn在旧index8和9之间，之后所有index整体+1
    old_map = list(range(9))      # 0..8 不变
    new_map = list(range(9))
    old_map += list(range(9, 15)) # 老的 9..14
    new_map += list(range(10, 16))# 对应新的 10..15 (因为9号位置插入了新attn)
    for oi, ni in zip(old_map, new_map):
        new_enc.conv_layers[ni].load_state_dict(old_enc.conv_layers[oi].state_dict())
    print(f"Encoder: 搬运了 {len(old_map)} 层，新增1层attention(32x32)为随机初始化")

def transplant_decoder(old_dec, new_dec):
    # 老decoder索引: 0=conv_in,1=res,2=attn(bottleneck),3=res,
    #               4-5=mul4 stage res,6=upsample,
    #               7-8=mul2 stage res,9=upsample,
    #               10-11=mul1 stage res,12=upsample,13=norm,14=conv_out,15=sigmoid
    # 新decoder在老index6(第一次upsample)之后插入新attn，之后所有index整体+1
    old_map = list(range(7))       # 0..6 不变
    new_map = list(range(7))
    old_map += list(range(7, 16))  # 老的 7..15
    new_map += list(range(8, 17))  # 对应新的 8..16
    for oi, ni in zip(old_map, new_map):
        new_dec.conv_layers[ni].load_state_dict(old_dec.conv_layers[oi].state_dict())
    print(f"Decoder: 搬运了 {len(old_map)} 层，新增1层attention(32x32)为随机初始化")


# ---------------- 4. 执行热启动 ----------------
old_ae = bld.BinaryAutoEncoder().to(bld.DEVICE)
old_ae.load_state_dict(
    torch.load('/content/drive/MyDrive/bld_purify_checkpoints/ae_taskaware_finetuned.pt',
               map_location=bld.DEVICE)
)

ae_attn = BinaryAutoEncoderAttn().to(bld.DEVICE)
transplant_encoder(old_ae.encoder, ae_attn.encoder)
transplant_decoder(old_ae.decoder, ae_attn.decoder)

# 关键修复：新插入的attention层(encoder idx9, decoder idx7)默认初始化不是恒等映射，
# 会在训练一开始就对热启动权重造成随机扰动。把proj层清零，让它们从"完全不改变输入"开始训练，
# 后续训练是纯增量式改进，不用先花epoch去抵消插入造成的干扰。
nn.init.zeros_(ae_attn.encoder.conv_layers[9].proj.weight)
nn.init.zeros_(ae_attn.encoder.conv_layers[9].proj.bias)
nn.init.zeros_(ae_attn.decoder.conv_layers[7].proj.weight)
nn.init.zeros_(ae_attn.decoder.conv_layers[7].proj.bias)
print("新增attention层已zero-init，训练起点等价于恒等映射")

print("热启动完成，ae_attn 已就绪，可以开始微调训练")

In [ ]:
"""
两阶段训练 ae_attn：
  Phase 1 — 冻结所有热启动层，只训练新插入的2个attention层（几个epoch，让它们从zero-init学出点东西）
  Phase 2 — 解冻全部，联合微调（跟之前finetune_taskaware一样的流程）
依赖：ae_attn (已完成zero-init热启动), finetune_taskaware, train_loader
"""

import torch

# ---------------- Phase 1：只训新增的2层attention ----------------
NEW_ATTN_MODULES = [ae_attn.encoder.conv_layers[9], ae_attn.decoder.conv_layers[7]]

for p in ae_attn.parameters():
    p.requires_grad_(False)
for m in NEW_ATTN_MODULES:
    for p in m.parameters():
        p.requires_grad_(True)

n_trainable = sum(p.numel() for p in ae_attn.parameters() if p.requires_grad)
print(f"Phase 1: 只训练新增attention层，可训练参数量 = {n_trainable}")
train_loader, _ = bld.get_loaders()
ae_attn = finetune_taskaware(ae_attn, train_loader, epochs=3, lambda_det=0.1, lr=3e-4)

# ---------------- Phase 2：解冻全部，联合微调 ----------------
for p in ae_attn.parameters():
    p.requires_grad_(True)

n_trainable = sum(p.numel() for p in ae_attn.parameters() if p.requires_grad)
print(f"Phase 2: 解冻全网络联合微调，可训练参数量 = {n_trainable}")

ae_attn = finetune_taskaware(ae_attn, train_loader, epochs=7, lambda_det=0.1, lr=1e-4)

torch.save(ae_attn.state_dict(), '/content/drive/MyDrive/bld_purify_checkpoints/ae_attn_taskaware_v2.pt')
print("两阶段训练完成，已存盘")

In [ ]:
import os
FT_ATTN_DIR = '/content/drive/MyDrive/dfire_adv_eval/ae_only_attn'
os.makedirs(f'{FT_ATTN_DIR}/images', exist_ok=True)
os.makedirs(f'{FT_ATTN_DIR}/labels', exist_ok=True)

ae_attn.eval()
n_done = 0
for x, targets, paths in tqdm(det_loader, desc="ae_only attn full eval"):
    x = x.to(bld.DEVICE)
    with torch.no_grad():
        x_128 = F.interpolate(x, size=128, mode='bilinear', align_corners=False)
        x_rec_128, _ = ae_attn(x_128)
    x_rec_384 = F.interpolate(x_rec_128, size=384, mode='bilinear', align_corners=False)
    save_as_dfire_split(x_rec_384, paths, FT_ATTN_DIR)
    n_done += len(paths)

map50, map5095 = eval_map(FT_ATTN_DIR, img_size=384)
print(f"\nae_only + attention (全量{n_done}张)  mAP50={map50:.4f}  mAP50-95={map5095:.4f}")
print("对照：")
print("  task-aware (无额外attn)  mAP50=0.1526")
print("  原始AE (纯MSE)           mAP50=0.0823")

In [ ]:
import os
V2_ATTN_DIR = '/content/drive/MyDrive/dfire_adv_eval/ae_only_attn_v2'
os.makedirs(f'{V2_ATTN_DIR}/images', exist_ok=True)
os.makedirs(f'{V2_ATTN_DIR}/labels', exist_ok=True)

ae_attn.eval()
n_done = 0
for x, targets, paths in tqdm(det_loader, desc="ae_only attn_v2 full eval"):
    x = x.to(bld.DEVICE)
    with torch.no_grad():
        x_128 = F.interpolate(x, size=128, mode='bilinear', align_corners=False)
        x_rec_128, _ = ae_attn(x_128)
    x_rec_384 = F.interpolate(x_rec_128, size=384, mode='bilinear', align_corners=False)
    save_as_dfire_split(x_rec_384, paths, V2_ATTN_DIR)
    n_done += len(paths)

map50, map5095 = eval_map(V2_ATTN_DIR, img_size=384)
print(f"\nae_only + attn(zero-init, 两阶段) 全量{n_done}张  mAP50={map50:.4f}  mAP50-95={map5095:.4f}")
print("对照：")
print("  task-aware (无attn)         mAP50=0.1526")
print("  attn (无zero-init, 5ep)      mAP50=0.1395")
print("  原始AE (纯MSE)               mAP50=0.0823")